# 03. Calidad de datos y análisis exploratorio

## Objetivo

Auditar y preparar el dataset maestro para el desarrollo de modelos de predicción
de incidencia de enfermedad hepática a 10 años.

En este notebook se analizarán:

- Integridad y estructura del dataset.
- Fuga temporal y fuga de información.
- Distribución de la variable objetivo.
- Valores ausentes.
- Códigos especiales de no respuesta.
- Variables constantes o cuasi-constantes.
- Outliers y valores clínicamente imposibles.
- Distribuciones y relaciones preliminares.
- Variables redundantes y correlacionadas.

Las transformaciones que dependen de los datos, como imputación, escalado,
balanceo o PCA, se ajustarán posteriormente utilizando únicamente el conjunto
de entrenamiento.

In [2]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RANDOM_STATE = 42
TARGET = "incident_liver_disease_10y"
DATASET_NAME = "klosa_liver_incident_10y_master"

In [3]:
def find_project_root(start_path: Path) -> Path:
    """
    Busca la raíz del proyecto identificando una carpeta que contenga
    simultáneamente los directorios 'data' y 'notebooks'.
    """
    start_path = start_path.resolve()

    candidate_paths = [start_path, *start_path.parents]

    for candidate in candidate_paths:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError(
        "No se ha encontrado la raíz del proyecto. "
        "Comprueba que existen las carpetas 'data' y 'notebooks'."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "data_quality"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Directorio de datos procesados: {PROCESSED_DIR}")
print(f"Directorio de informes: {REPORTS_DIR}")

Raíz del proyecto: C:\Users\DAVID\TFM_Liver_Disease_Risk
Directorio de datos procesados: C:\Users\DAVID\TFM_Liver_Disease_Risk\data\processed
Directorio de informes: C:\Users\DAVID\TFM_Liver_Disease_Risk\reports\data_quality


In [4]:
SUPPORTED_EXTENSIONS = [".csv", ".xlsx", ".xls", ".parquet"]

dataset_candidates = []

for extension in SUPPORTED_EXTENSIONS:
    dataset_candidates.extend(
        DATA_DIR.rglob(f"{DATASET_NAME}{extension}")
    )

dataset_candidates = sorted(set(dataset_candidates))

if not dataset_candidates:
    raise FileNotFoundError(
        f"No se ha encontrado '{DATASET_NAME}' dentro de {DATA_DIR}"
    )

print("Archivos encontrados:")

for index, path in enumerate(dataset_candidates, start=1):
    print(f"{index}. {path}")

DATASET_PATH = dataset_candidates[0]

print(f"\nDataset seleccionado: {DATASET_PATH}")

Archivos encontrados:
1. C:\Users\DAVID\TFM_Liver_Disease_Risk\data\processed\klosa_liver_incident_10y_master.csv
2. C:\Users\DAVID\TFM_Liver_Disease_Risk\data\processed\klosa_liver_incident_10y_master.parquet

Dataset seleccionado: C:\Users\DAVID\TFM_Liver_Disease_Risk\data\processed\klosa_liver_incident_10y_master.csv


In [5]:
def load_dataset(path: Path) -> pd.DataFrame:
    """Carga un dataset según su extensión."""

    suffix = path.suffix.lower()

    if suffix == ".csv":
        try:
            return pd.read_csv(path, encoding="utf-8", low_memory=False)
        except UnicodeDecodeError:
            return pd.read_csv(path, encoding="latin-1", low_memory=False)

    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    if suffix == ".parquet":
        return pd.read_parquet(path)

    raise ValueError(f"Formato no soportado: {suffix}")


df = load_dataset(DATASET_PATH)
df_raw = df.copy(deep=True)

print("Dataset cargado correctamente.")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")

Dataset cargado correctamente.
Filas: 6,560
Columnas: 310


In [6]:
if TARGET not in df.columns:
    target_candidates = [
        column
        for column in df.columns
        if "liver" in column.lower()
        or "incident" in column.lower()
        or "target" in column.lower()
    ]

    raise KeyError(
        f"No se ha encontrado la variable objetivo '{TARGET}'.\n"
        f"Posibles candidatas: {target_candidates}"
    )

print(f"Variable objetivo encontrada: {TARGET}")

target_values = sorted(
    df[TARGET].dropna().unique().tolist()
)

print(f"Valores encontrados: {target_values}")

if not set(target_values).issubset({0, 1}):
    print(
        "ADVERTENCIA: la variable objetivo contiene valores diferentes de 0 y 1."
    )

Variable objetivo encontrada: incident_liver_disease_10y
Valores encontrados: [0, 1]


1. Comprobamos filas duplicadas que puedan interferir en la calidad de los datos


In [7]:
n_duplicate_rows = df.duplicated().sum()

print(f"Filas completamente duplicadas: {n_duplicate_rows:,}")
print(f"Porcentaje: {n_duplicate_rows / len(df) * 100:.3f}%")

Filas completamente duplicadas: 0
Porcentaje: 0.000%


2. Detectamos variables cuasi-constantes, donde un único valor representa más del 99 %:

In [8]:
unique_counts = df.nunique(dropna=False).sort_values()

constant_columns = unique_counts[
    unique_counts <= 1
].index.tolist()

print(f"Variables constantes: {len(constant_columns)}")
print(constant_columns)

Variables constantes: 2
['mniw_y', 'g013']


In [9]:
quasi_constant_report = []

for column in df.columns:
    frequencies = df[column].value_counts(
        normalize=True,
        dropna=False
    )

    if len(frequencies) == 0:
        continue

    dominant_value = frequencies.index[0]
    dominant_frequency = frequencies.iloc[0]

    if dominant_frequency >= 0.99:
        quasi_constant_report.append({
            "variable": column,
            "valor_dominante": dominant_value,
            "porcentaje_dominante": dominant_frequency * 100,
            "valores_unicos": df[column].nunique(dropna=False)
        })

quasi_constant_report = pd.DataFrame(
    quasi_constant_report
).sort_values(
    "porcentaje_dominante",
    ascending=False
)

display(quasi_constant_report)

,variable,valor_dominante,porcentaje_dominante,valores_unicos
21,g013,NaN,100.000,1
24,mniw_y,"2,006.000",100.000,1
1,a035_07,NaN,99.985,2
22,g014,NaN,99.985,2
23,industrial,NaN,99.939,4
4,businessfarm,NaN,99.909,7
27,unemployment,NaN,99.817,11
26,personal,NaN,99.771,16
17,d_com108,NaN,99.695,7
20,g012,NaN,99.466,10


3. Buscaremos nombres que parezcan pertenecer a olas posteriores:

In [10]:
future_wave_patterns = [
    r"(^|_)w0?[2-9]($|_)",
    r"wave[_ ]?[2-9]",
    r"ola[_ ]?[2-9]",
    r"follow.?up",
    r"incident",
    r"event",
    r"diagnos",
    r"liver",
    r"hepatic",
    r"onset",
    r"develop"
]

potential_leakage_columns = []

for column in df.columns:
    column_lower = column.lower()

    matched_patterns = [
        pattern
        for pattern in future_wave_patterns
        if re.search(pattern, column_lower)
    ]

    if matched_patterns:
        potential_leakage_columns.append({
            "variable": column,
            "patrones_detectados": ", ".join(matched_patterns),
            "valores_unicos": df[column].nunique(dropna=True),
            "porcentaje_nulos": df[column].isna().mean() * 100
        })

potential_leakage_report = pd.DataFrame(
    potential_leakage_columns
).sort_values("variable")

display(potential_leakage_report)

,variable,patrones_detectados,valores_unicos,porcentaje_nulos
0,incident_liver_disease_10y,"incident, liver",2,0.000


4. Balance de clases

In [11]:
target_distribution = (
    df[TARGET]
    .value_counts(dropna=False)
    .rename_axis("clase")
    .reset_index(name="frecuencia")
)

target_distribution["porcentaje"] = (
    target_distribution["frecuencia"] / len(df) * 100
)

display(target_distribution)

,clase,frecuencia,porcentaje
0,0,6429,98.003
1,1,131,1.997


5. Valores perdidos (missing values)

In [12]:
missing_summary = pd.DataFrame({
    "nulos": df.isna().sum(),
    "porcentaje_nulos": df.isna().mean().mul(100),
    "tipo": df.dtypes.astype(str),
    "valores_unicos": df.nunique(dropna=True)
})

missing_summary = (
    missing_summary
    .query("nulos > 0")
    .sort_values("porcentaje_nulos", ascending=False)
)

display(missing_summary)

,nulos,porcentaje_nulos,tipo,valores_unicos
g013,6560,100.000,float64,0
g014,6559,99.985,float64,1
a035_07,6559,99.985,float64,1
industrial,6556,99.939,float64,3
businessfarm,6554,99.909,float64,6
...,...,...,...,...
c145,42,0.640,float64,4
c142,42,0.640,float64,4
c149,42,0.640,float64,4
c081,20,0.305,float64,2


In [13]:
missing_bands = pd.cut(
    df.isna().mean().mul(100),
    bins=[-0.01, 0, 5, 20, 40, 60, 80, 100],
    labels=[
        "0%",
        "0-5%",
        "5-20%",
        "20-40%",
        "40-60%",
        "60-80%",
        "80-100%"
    ]
)

display(
    missing_bands
    .value_counts()
    .sort_index()
    .rename("numero_variables")
    .to_frame()
)

,numero_variables
0%,93
0-5%,14
5-20%,7
20-40%,6
40-60%,40
60-80%,28
80-100%,122


6. Comprobación de ausencia entre clases

In [14]:
target_classes = sorted(df[TARGET].dropna().unique())

missing_by_target = pd.DataFrame({
    f"missing_clase_{target_class}": (
        df.loc[df[TARGET] == target_class]
        .isna()
        .mean()
        .mul(100)
    )
    for target_class in target_classes
})

if len(target_classes) == 2:
    class_0 = target_classes[0]
    class_1 = target_classes[1]

    missing_by_target["diferencia_absoluta"] = (
        missing_by_target[f"missing_clase_{class_1}"]
        - missing_by_target[f"missing_clase_{class_0}"]
    ).abs()

    missing_by_target = missing_by_target.sort_values(
        "diferencia_absoluta",
        ascending=False
    )

display(missing_by_target.head(30))

,missing_clase_0,missing_clase_1,diferencia_absoluta
chronic_j,58.345,41.985,16.360
c068,41.655,58.015,16.360
labor_st,47.146,59.542,12.396
retired,47.146,59.542,12.396
residence,47.955,36.641,11.313
c312,62.918,73.282,10.364
c311,62.918,73.282,10.364
residence_,20.314,30.534,10.220
a035_02,41.002,51.145,10.143
c303,41.873,51.908,10.036


7. Detección de códigos espceiales

In [15]:
suspect_codes = [
    -999999, -99999, -9999, -999, -99, -9,
    -8, -7, -1,
    97, 98, 99,
    997, 998, 999,
    9997, 9998, 9999,
    99997, 99998, 99999,
    999997, 999998, 999999
]

special_code_report = []

numeric_columns = df.select_dtypes(
    include=np.number
).columns

for column in numeric_columns:
    value_counts = df[column].value_counts(dropna=False)

    for code in suspect_codes:
        if code in value_counts.index:
            count = int(value_counts.loc[code])

            special_code_report.append({
                "variable": column,
                "codigo_sospechoso": code,
                "frecuencia": count,
                "porcentaje": count / len(df) * 100
            })

special_code_report = pd.DataFrame(special_code_report)

if not special_code_report.empty:
    special_code_report = special_code_report.sort_values(
        ["variable", "codigo_sospechoso"]
    )

display(special_code_report)

,variable,codigo_sospechoso,frecuencia,porcentaje
0,a002_age,97,1,0.015
1,a030,97,55,0.838
2,agriculture,-9,15,0.229
3,agriculture,-8,4,0.061
4,annuity,-8,3,0.046
5,annuity,99,1,0.015
6,assetinc,-9,156,2.378
7,ba068,-9,6,0.091
8,ba068,-8,2,0.030
9,businessfarm,-9,1,0.015


In [16]:
feature_columns = [
    column
    for column in df.columns
    if column != TARGET
]

numeric_features = (
    df[feature_columns]
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

non_numeric_features = [
    column
    for column in feature_columns
    if column not in numeric_features
]

low_cardinality_numeric = [
    column
    for column in numeric_features
    if df[column].nunique(dropna=True) <= 15
]

continuous_numeric_candidates = [
    column
    for column in numeric_features
    if df[column].nunique(dropna=True) > 15
]

print(f"Variables predictoras totales: {len(feature_columns)}")
print(f"Variables numéricas: {len(numeric_features)}")
print(f"Variables no numéricas: {len(non_numeric_features)}")
print(
    "Variables numéricas posiblemente categóricas: "
    f"{len(low_cardinality_numeric)}"
)
print(
    "Variables numéricas posiblemente continuas: "
    f"{len(continuous_numeric_candidates)}"
)

Variables predictoras totales: 309
Variables numéricas: 309
Variables no numéricas: 0
Variables numéricas posiblemente categóricas: 221
Variables numéricas posiblemente continuas: 88


In [17]:
categorical_candidate_report = pd.DataFrame({
    "variable": low_cardinality_numeric,
    "valores_unicos": [
        df[column].nunique(dropna=True)
        for column in low_cardinality_numeric
    ],
    "valores": [
        sorted(df[column].dropna().unique().tolist())
        for column in low_cardinality_numeric
    ]
})

display(categorical_candidate_report)

,variable,valores_unicos,valores
0,a002m,12,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]"
1,a003,2,"[1, 5]"
2,a030,6,"[1, 2, 3, 4, 5, 97]"
3,a032,10,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"
4,a035_01,10,"[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]"
...,...,...,...
216,size_c,11,"[-9.0, -8.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]"
217,smoke,3,"[0, 1, 2]"
218,status,6,"[-8.0, 1.0, 3.0, 5.0, 7.0, 9.0]"
219,target1,2,"[1.0, 2.0]"


In [18]:
summary_initial = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "no_nulos": df.notna().sum(),
    "nulos": df.isna().sum(),
    "porcentaje_nulos": df.isna().mean().mul(100),
    "valores_unicos": df.nunique(dropna=True)
}).sort_values(
    ["porcentaje_nulos", "valores_unicos"],
    ascending=[False, True]
)

display(summary_initial)

,tipo,no_nulos,nulos,porcentaje_nulos,valores_unicos
g013,float64,0,6560,100.000,0
a035_07,float64,1,6559,99.985,1
g014,float64,1,6559,99.985,1
industrial,float64,4,6556,99.939,3
businessfarm,float64,6,6554,99.909,6
...,...,...,...,...,...
year2,int64,6560,0,0.000,62
c105,int64,6560,0,0.000,70
hhinc,int64,6560,0,0.000,263
wgt_c,float64,6560,0,0.000,930


In [19]:
summary_initial.to_csv(
    REPORTS_DIR / "initial_variable_summary.csv",
    encoding="utf-8-sig"
)

missing_summary.to_csv(
    REPORTS_DIR / "missing_values_summary.csv",
    encoding="utf-8-sig"
)

quasi_constant_report.to_csv(
    REPORTS_DIR / "quasi_constant_variables.csv",
    index=False,
    encoding="utf-8-sig"
)

potential_leakage_report.to_csv(
    REPORTS_DIR / "potential_data_leakage.csv",
    index=False,
    encoding="utf-8-sig"
)

special_code_report.to_csv(
    REPORTS_DIR / "special_codes_report.csv",
    index=False,
    encoding="utf-8-sig"
)

target_distribution.to_csv(
    REPORTS_DIR / "target_distribution.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Informes guardados en: {REPORTS_DIR}")

Informes guardados en: C:\Users\DAVID\TFM_Liver_Disease_Risk\reports\data_quality


## Tratamiento de los códigos especiales de KLoSA

La documentación de KLoSA indica que el cuestionario utiliza rutas automáticas,
respuestas de no conocimiento o rechazo, preguntas no aplicables y archivos
específicos de imputación de no respuesta.

Los códigos negativos `-8` y `-9` no representan valores reales de las
variables. Sin embargo, su significado concreto puede variar según la variable
y la ola del estudio.

Por este motivo:

- No se eliminarán pacientes que contengan estos códigos.
- No se eliminarán variables únicamente por contenerlos.
- No se considerarán valores numéricos válidos.
- Se sustituirán por valores ausentes en una copia de trabajo.
- La imputación se ajustará posteriormente utilizando solo el conjunto de
  entrenamiento.
- Se conservará la posibilidad de añadir indicadores de ausencia dentro del
  pipeline de preprocesamiento.

Otros códigos como 97, 98, 99, 9997 o 9999 no se modificarán globalmente,
ya que pueden representar valores o categorías válidas dependiendo de la
variable.


In [20]:
df_work = df.copy(deep=True)

ID_COLUMN = "pid"
WEIGHT_COLUMN = "wgt_c"
SPECIAL_MISSING_CODES = [-8, -9]

print(f"Dataset maestro: {df.shape}")
print(f"Copia de trabajo: {df_work.shape}")

Dataset maestro: (6560, 310)
Copia de trabajo: (6560, 310)


In [21]:
numeric_columns = (
    df_work
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

special_code_records = []

for column in numeric_columns:
    for code in SPECIAL_MISSING_CODES:
        mask = df_work[column].eq(code)
        frequency = int(mask.sum())

        if frequency == 0:
            continue

        class_0_count = int(
            (
                mask
                & df_work[TARGET].eq(0)
            ).sum()
        )

        class_1_count = int(
            (
                mask
                & df_work[TARGET].eq(1)
            ).sum()
        )

        special_code_records.append({
            "variable": column,
            "codigo": code,
            "frecuencia": frequency,
            "porcentaje_dataset": frequency / len(df_work) * 100,
            "casos_clase_0": class_0_count,
            "casos_clase_1": class_1_count,
            "valores_unicos_incluyendo_codigo": (
                df_work[column].nunique(dropna=True)
            ),
            "nulos_originales": int(
                df_work[column].isna().sum()
            )
        })

special_missing_audit = (
    pd.DataFrame(special_code_records)
    .sort_values(
        ["frecuencia", "variable"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

display(special_missing_audit)

,variable,codigo,frecuencia,porcentaje_dataset,casos_clase_0,casos_clase_1,valores_unicos_incluyendo_codigo,nulos_originales
0,c124m,-9,1994,30.396,1946,48,14,3646
1,c306,-9,1960,29.878,1931,29,55,367
2,job_startm,-9,1318,20.091,1289,29,14,3728
3,c007m,-9,871,13.277,854,17,14,4843
4,passets,-9,825,12.576,815,10,1168,1418
5,pnetassets,-9,803,12.241,792,11,1300,1287
6,d_com139m,-9,540,8.232,530,10,13,5462
7,c049m,-9,536,8.171,528,8,13,5508
8,financialasset,-9,469,7.149,465,4,622,3237
9,realestate,-9,375,5.716,369,6,225,2921


In [22]:
special_code_summary = (
    special_missing_audit
    .groupby("codigo", as_index=False)
    .agg(
        frecuencia_total=("frecuencia", "sum"),
        variables_afectadas=("variable", "nunique"),
        casos_clase_0=("casos_clase_0", "sum"),
        casos_clase_1=("casos_clase_1", "sum")
    )
)

special_code_summary["porcentaje_sobre_celdas_numericas"] = (
    special_code_summary["frecuencia_total"]
    / df_work[numeric_columns].size
    * 100
)

display(special_code_summary)

,codigo,frecuencia_total,variables_afectadas,casos_clase_0,casos_clase_1,porcentaje_sobre_celdas_numericas
0,-9,13288,120,13042,246,0.653
1,-8,1534,55,1498,36,0.075


In [23]:
special_mask = (
    df_work[numeric_columns]
    .isin(SPECIAL_MISSING_CODES)
)

special_codes_by_patient = pd.DataFrame({
    ID_COLUMN: df_work[ID_COLUMN],
    TARGET: df_work[TARGET],
    "numero_codigos_especiales": special_mask.sum(axis=1),
    "tiene_codigo_especial": special_mask.any(axis=1)
})

n_affected_patients = int(
    special_codes_by_patient["tiene_codigo_especial"].sum()
)

print(
    "Pacientes con al menos un -8 o -9: "
    f"{n_affected_patients:,} de {len(df_work):,} "
    f"({n_affected_patients / len(df_work) * 100:.2f}%)"
)

display(
    special_codes_by_patient["numero_codigos_especiales"]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

Pacientes con al menos un -8 o -9: 4,709 de 6,560 (71.78%)


count   6,560.000
mean        2.259
std         2.738
min         0.000
25%         0.000
50%         1.000
75%         3.000
90%         6.000
95%         7.000
99%        11.000
max        41.000
Name: numero_codigos_especiales, dtype: float64

In [24]:
special_code_columns = [
    column
    for column in numeric_columns
    if df_work[column]
    .isin(SPECIAL_MISSING_CODES)
    .any()
]

print(
    f"Variables que contienen -8 o -9: "
    f"{len(special_code_columns)}"
)

print(special_code_columns)

Variables que contienen -8 o -9: 131
['agriculture', 'annuity', 'assetinc', 'ba068', 'businessfarm', 'c007m', 'c007y', 'c012m', 'c012y', 'c017m', 'c024m', 'c024y', 'c034m', 'c034y', 'c039m', 'c039y', 'c044m', 'c044y', 'c049m', 'c049y', 'c064m', 'c068', 'c105', 'c107', 'c124m', 'c124y', 'c126', 'c127', 'c301', 'c302', 'c303', 'c306', 'c309', 'c310', 'c311', 'c312', 'c330', 'c337', 'c340', 'chronic_j', 'd_com001', 'd_com014', 'd_com015', 'd_com024', 'd_com025', 'd_com031', 'd_com032', 'd_com037', 'd_com049', 'd_com052', 'd_com053', 'd_com054', 'd_com055', 'd_com056', 'd_com057', 'd_com066', 'd_com073', 'd_com074', 'd_com075', 'd_com076', 'd_com077', 'd_com078', 'd_com079', 'd_com080', 'd_com081', 'd_com082', 'd_com084', 'd_com085', 'd_com089', 'd_com090', 'd_com092', 'd_com093', 'd_com095', 'd_com096', 'd_com101', 'd_com102', 'd_com125', 'd_com139m', 'd_com139y', 'd_com142', 'earned', 'ecoact_s', 'edu_s', 'exploit', 'f001type', 'f013_n', 'financial', 'financialasset', 'fromchildren', 'fr

In [29]:
# Restaurar la copia para evitar una transformación parcial
df_work = df.copy(deep=True)

ID_COLUMN = "pid"
WEIGHT_COLUMN = "wgt_c"
SPECIAL_MISSING_CODES = [-8, -9]

# Seleccionar únicamente columnas numéricas
numeric_columns = (
    df_work
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

# Columnas que contienen al menos un -8 o -9
special_code_columns = [
    column
    for column in numeric_columns
    if df_work[column]
    .isin(SPECIAL_MISSING_CODES)
    .any()
]

special_values_before = int(
    df_work[special_code_columns]
    .isin(SPECIAL_MISSING_CODES)
    .sum()
    .sum()
)

conversion_records = []

# Transformación columna a columna
for column in special_code_columns:
    original_dtype = str(df_work[column].dtype)

    series = pd.to_numeric(
        df_work[column],
        errors="coerce"
    ).astype("float64")

    special_mask = series.isin(
        SPECIAL_MISSING_CODES
    )

    number_replaced = int(
        special_mask.sum()
    )

    df_work[column] = series.mask(
        special_mask,
        np.nan
    )

    conversion_records.append({
        "variable": column,
        "tipo_original": original_dtype,
        "tipo_resultante": str(df_work[column].dtype),
        "valores_reemplazados": number_replaced
    })

special_values_after = int(
    df_work[special_code_columns]
    .isin(SPECIAL_MISSING_CODES)
    .sum()
    .sum()
)

print(f"Columnas afectadas: {len(special_code_columns)}")
print(f"Códigos especiales antes: {special_values_before:,}")
print(f"Códigos especiales después: {special_values_after:,}")

Columnas afectadas: 131
Códigos especiales antes: 14,822
Códigos especiales después: 0


In [30]:
missing_after_special_codes = pd.DataFrame({
    "tipo": df_work.dtypes.astype(str),
    "nulos_totales": df_work.isna().sum(),
    "porcentaje_nulos": (
        df_work.isna().mean() * 100
    ),
    "valores_unicos_observados": (
        df_work.nunique(dropna=True)
    )
}).sort_values(
    "porcentaje_nulos",
    ascending=False
)

display(missing_after_special_codes.head(40))

,tipo,nulos_totales,porcentaje_nulos,valores_unicos_observados
g013,float64,6560,100.000,0
g014,float64,6559,99.985,1
a035_07,float64,6559,99.985,1
industrial,float64,6556,99.939,3
businessfarm,float64,6555,99.924,5
unemployment,float64,6549,99.832,9
personal,float64,6545,99.771,15
d_com108,float64,6540,99.695,6
size_c,float64,6536,99.634,9
d_com102,float64,6526,99.482,16


In [32]:
missing_comparison = pd.DataFrame({
    "nulos_antes": df.isna().sum(),
    "nulos_despues": df_work.isna().sum()
})

missing_comparison["nuevos_nulos_por_codigos"] = (
    missing_comparison["nulos_despues"]
    - missing_comparison["nulos_antes"]
)

missing_comparison["porcentaje_nulos_despues"] = (
    df_work.isna().mean() * 100
)

missing_comparison = (
    missing_comparison
    .query("nuevos_nulos_por_codigos > 0")
    .sort_values(
        "nuevos_nulos_por_codigos",
        ascending=False
    )
)

display(missing_comparison.head(40))

,nulos_antes,nulos_despues,nuevos_nulos_por_codigos,porcentaje_nulos_despues
c124m,3646,5641,1995,85.991
c306,367,2356,1989,35.915
job_startm,3728,5052,1324,77.012
c007m,4843,5715,872,87.119
pnetassets,1287,2135,848,32.546
passets,1418,2243,825,34.192
d_com139m,5462,6002,540,91.494
c049m,5508,6044,536,92.134
financialasset,3237,3706,469,56.494
hhinc,0,431,431,6.570


In [33]:
special_missing_audit.to_csv(
    REPORTS_DIR / "special_missing_codes_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

special_code_summary.to_csv(
    REPORTS_DIR / "special_missing_codes_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

special_codes_by_patient.to_csv(
    REPORTS_DIR / "special_missing_codes_by_patient.csv",
    index=False,
    encoding="utf-8-sig"
)

missing_after_special_codes.to_csv(
    REPORTS_DIR / "missing_values_after_special_codes.csv",
    encoding="utf-8-sig"
)

missing_comparison.to_csv(
    REPORTS_DIR / "missing_values_before_after_special_codes.csv",
    encoding="utf-8-sig"
)

print("Informes sobre códigos especiales guardados correctamente.")

Informes sobre códigos especiales guardados correctamente.


In [34]:
protected_columns = {
    TARGET,
    ID_COLUMN,
    WEIGHT_COLUMN
}

no_information_columns = [
    column
    for column in df_work.columns
    if (
        column not in protected_columns
        and df_work[column].nunique(dropna=True) <= 1
    )
]

no_information_report = pd.DataFrame({
    "variable": no_information_columns,
    "valores_unicos_observados": [
        df_work[column].nunique(dropna=True)
        for column in no_information_columns
    ],
    "nulos": [
        df_work[column].isna().sum()
        for column in no_information_columns
    ],
    "porcentaje_nulos": [
        df_work[column].isna().mean() * 100
        for column in no_information_columns
    ],
    "valor_observado": [
        (
            df_work[column].dropna().iloc[0]
            if df_work[column].notna().any()
            else np.nan
        )
        for column in no_information_columns
    ]
})

display(no_information_report)

,variable,valores_unicos_observados,nulos,porcentaje_nulos,valor_observado
0,a035_07,1,6559,99.985,5.000
1,bb_adl_num3,1,6522,99.421,1.000
2,g013,0,6560,100.000,NaN
3,g014,1,6559,99.985,70.000
4,mniw_y,1,0,0.000,"2,006.000"


In [35]:
df_stage1 = df_work.drop(
    columns=no_information_columns
).copy()

print("PRIMERA DEPURACIÓN SEGURA")
print("=" * 50)
print(f"Dimensiones originales: {df.shape}")
print(
    f"Variables sin información eliminadas: "
    f"{len(no_information_columns)}"
)
print(f"Dimensiones resultantes: {df_stage1.shape}")

PRIMERA DEPURACIÓN SEGURA
Dimensiones originales: (6560, 310)
Variables sin información eliminadas: 5
Dimensiones resultantes: (6560, 305)


In [36]:
no_information_report.to_csv(
    REPORTS_DIR / "excluded_no_information_variables.csv",
    index=False,
    encoding="utf-8-sig"
)

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

STAGE1_PATH = (
    INTERIM_DIR
    / "klosa_liver_stage1_special_codes_cleaned.csv"
)

df_stage1.to_csv(
    STAGE1_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(f"Dataset provisional guardado en: {STAGE1_PATH}")
print(f"Dimensiones: {df_stage1.shape}")

Dataset provisional guardado en: C:\Users\DAVID\TFM_Liver_Disease_Risk\data\interim\klosa_liver_stage1_special_codes_cleaned.csv
Dimensiones: (6560, 305)
